# Flatmate RL GRPO 2: Step-by-Step Curriculum

This notebook trains the Flatmate RL broker policy with a step curriculum:

1. collect web-endpoint rollouts from the heuristic policy
2. keep `scenario_id`, `seed`, `step_idx`, `prefix_actions_json`, and `reference_action_json` for every state
3. evaluate expected response vs generated response by step
4. train first on step 1 examples from different seeds
5. expand to steps 1-2, then 1-3, and so on while keeping the same learned adapter

The reward combines:

- JSON/schema validity
- expected-action accuracy for the current step
- live web endpoint transition reward after replaying the stored prefix

This is meant for tracking step-by-step accuracy improvement, not only final episode reward.


In [1]:
# Restart the kernel after this cell if your notebook runtime asks you to.
%pip install -q "trl>=0.26.2" "transformers>=4.57.0" accelerate datasets peft websockets huggingface_hub matplotlib pandas tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import asyncio
import gc
import inspect
import json
import math
import os
import random
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import pandas as pd
import torch
import websockets
from datasets import Dataset
from peft import AutoPeftModelForCausalLM, LoraConfig, PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer

from env_config import load_repo_env

load_repo_env()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

SPACE_HTTP_URL = "https://kushalexplores-flatmate-rl.hf.space"
SCENARIOS = [
    "task_visit_single",
    "task_visit_single_hidden_flex",
    "task_visit_multi",
    "task_visit_single_seller_followup",
]

MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen3-4B-Instruct-2507")
PREVIOUS_GRPO_DIR = os.getenv("PREVIOUS_GRPO_DIR", "")
PREVIOUS_SFT_DIR = os.getenv("PREVIOUS_SFT_DIR", "")
OUTPUT_DIR = "flatmate-rl-grpo-2-step-curriculum_priyanka"
SFT_OUTPUT_DIR = f"{OUTPUT_DIR}/sft_step_warmup"
RESULTS_DIR = Path(OUTPUT_DIR) / "step_metrics"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS_PER_SCENARIO = 5
START_SEED = 101
MAX_COLLECT_STEPS = 14

# Toggle pilot/full run from env: PILOT_MODE=true|false
PILOT_MODE = os.getenv("PILOT_MODE", "true").strip().lower() in {"1", "true", "yes", "y", "on"}

if PILOT_MODE:
    MAX_CURRICULUM_STEP = 1
    NUM_GENERATIONS = 1
    GRPO_STEPS_PER_STAGE = 12
    EVAL_ATTEMPTS = 2
    GRPO_BETA = 0.0
else:
    MAX_CURRICULUM_STEP = 6  # 1-based. Increase after the loop is stable.
    NUM_GENERATIONS = 3
    GRPO_STEPS_PER_STAGE = 120
    EVAL_ATTEMPTS = 5
    GRPO_BETA = 0.02

# L4-safe defaults for Qwen3-4B GRPO.
MAX_PROMPT_LENGTH = 768
MAX_COMPLETION_LENGTH = 96
SFT_MAX_LENGTH = 768
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 12
GRPO_BATCH_SIZE = 1
GRPO_GRAD_ACCUM = 16

SFT_EPOCHS = 2

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

print(
    f"Pilot mode={PILOT_MODE} | stages={MAX_CURRICULUM_STEP} | "
    f"grpo_steps_per_stage={GRPO_STEPS_PER_STAGE} | "
    f"num_generations={NUM_GENERATIONS} | eval_attempts={EVAL_ATTEMPTS}"
)
print(
    f"Memory profile: prompt={MAX_PROMPT_LENGTH}, completion={MAX_COMPLETION_LENGTH}, "
    f"sft_bs={SFT_BATCH_SIZE}, grpo_bs={GRPO_BATCH_SIZE}, beta={GRPO_BETA}"
)

def ws_url_from_http(base_url: str) -> str:
    parsed = urlparse(base_url.rstrip("/"))
    scheme = "wss" if parsed.scheme == "https" else "ws"
    return f"{scheme}://{parsed.netloc}/ws"

SPACE_WS_URL = ws_url_from_http(SPACE_HTTP_URL)
SPACE_WS_URL


'wss://kushalexplores-flatmate-rl.hf.space/ws'

## Web Endpoint Client

The notebook uses the live web endpoint. Every reward replay opens a websocket, resets to the stored scenario/seed, replays the stored prefix actions, then applies the candidate model action.


In [3]:
class FlatmateEndpoint:
    def __init__(self, ws_url: str = SPACE_WS_URL, timeout_s: float = 120.0):
        self.ws_url = ws_url
        self.timeout_s = timeout_s

    async def __aenter__(self):
        self.ws = await websockets.connect(
            self.ws_url,
            open_timeout=self.timeout_s,
            ping_timeout=self.timeout_s,
        )
        return self

    async def __aexit__(self, exc_type, exc, tb):
        try:
            await self.ws.send(json.dumps({"type": "close"}))
        finally:
            await self.ws.close()

    async def _send(self, payload: dict[str, Any]) -> dict[str, Any]:
        await self.ws.send(json.dumps(payload))
        raw = await asyncio.wait_for(self.ws.recv(), timeout=self.timeout_s)
        message = json.loads(raw)
        if message.get("type") == "error":
            raise RuntimeError(message.get("data", message))
        data = message["data"]
        obs = data.get("observation", {})
        obs["reward"] = data.get("reward")
        obs["done"] = data.get("done", False)
        return obs

    async def reset(self, scenario_id: str, seed: int | None = None) -> dict[str, Any]:
        data: dict[str, Any] = {"scenario_id": scenario_id}
        if seed is not None:
            data["seed"] = seed
        return await self._send({"type": "reset", "data": data})

    async def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return await self._send({"type": "step", "data": action})

async def smoke_test_endpoint():
    async with FlatmateEndpoint() as env:
        obs = await env.reset("task_visit_single", seed=START_SEED)
        return {"scenario_id": obs["scenario_id"], "status": obs["status"], "message": obs.get("last_user_message") or obs.get("current_user_request")}

await smoke_test_endpoint()


{'scenario_id': 'task_visit_single',
 'status': 'ready',
 'message': "Hi, I'm looking for a flatmate-share near Goregaon East. My budget is up to Rs. 20,000 and I'm mainly considering Andheri West or Jogeshwari because I work as a software engineer at a startup."}

## Heuristic Policy and Prompts

The heuristic defines the expected response for each state. These expected actions are used for SFT targets, expected-action reward, and step accuracy metrics.


In [4]:
def trace_tool_names(obs: dict[str, Any]) -> list[str]:
    return [str(t.get("tool", t.get("tool_name", ""))) for t in obs.get("tool_trace", [])]

def heuristic_action(obs: dict[str, Any]) -> dict[str, Any] | None:
    if obs.get("done"):
        return None

    tool_names = trace_tool_names(obs)
    phase = obs.get("phase", "buyer")
    remaining = set(obs.get("remaining_required_fields", []))
    scenario_id = obs.get("scenario_id", "task_visit_single")
    booked = [str(item.get("post_id", "")) for item in obs.get("booked_visits", [])]
    selected_posts = list(obs.get("selected_posts", []))
    last_user_message = str(obs.get("last_user_message", "")).lower()
    buyer_history = obs.get("buyer_conversation_history", [])
    last_role = str(buyer_history[-1].get("role", "")) if buyer_history else ""
    user_has_replied = obs.get("status") == "user_response" and last_role == "user"

    def has(name: str) -> bool:
        return name in tool_names

    def count(name: str) -> int:
        return tool_names.count(name)

    def ask_for_buyer_fields() -> dict[str, Any] | None:
        if "diet" in remaining and "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference and visit availability."}
        if "diet" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference."}
        if "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your visit availability."}
        return None

    def ask_for_seller_fields() -> dict[str, Any] | None:
        need_dietary = "dietary" in remaining
        need_occupation = "occupation_requirement" in remaining
        need_slots = "calendar_slots" in remaining
        if need_dietary and need_occupation and need_slots:
            return {"action_type": "assistant_message", "assistant_message": "Please share the household dietary setup, who the flat is for, and available visit time slots."}
        if need_dietary or need_occupation or need_slots:
            return {"action_type": "assistant_message", "assistant_message": "Please share the remaining seller details (dietary setup, who the flat is for, available visit time slots)."}
        return None

    if phase == "seller":
        if not obs.get("seller_profile_stored"):
            ask = ask_for_seller_fields()
            if ask is not None:
                return ask
            return {"action_type": "tool_call", "tool_name": "store_seller_details", "tool_arguments": {}}
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_dynamic_followup_1"]}}
        if not has("check_table_slot_matches"):
            return {"action_type": "tool_call", "tool_name": "check_table_slot_matches", "tool_arguments": {"post_ids": ["post_dynamic_followup_1"]}}
        if not has("confirm_seller_match"):
            return {"action_type": "tool_call", "tool_name": "confirm_seller_match", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}
        if not has("offer_matched_listing_to_buyer"):
            return {"action_type": "tool_call", "tool_name": "offer_matched_listing_to_buyer", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}
        return {"action_type": "tool_call", "tool_name": "schedule_table_visit", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}

    if scenario_id == "task_visit_single_seller_followup" and not obs.get("seller_profile_stored"):
        if not obs.get("buyer_profile_stored"):
            ask = ask_for_buyer_fields()
            if ask is not None:
                return ask
            return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}
        if not has("search_posts"):
            return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}
        return {"action_type": "tool_call", "tool_name": "close_buyer_conversation", "tool_arguments": {}}

    if not obs.get("buyer_profile_stored"):
        ask = ask_for_buyer_fields()
        if ask is not None:
            return ask
        return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}

    if not has("search_posts"):
        return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}

    if scenario_id == "task_visit_single":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_023", "post_031"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_023", "post_031"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_023"]}}
        if not has("contact_poster") and not has("book_viewing") and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "post_023 is available Saturday 11am. Please confirm Saturday 11am if that works."}
        if not has("contact_poster"):
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_023", "time_text": "Saturday 11am"}}
        if not has("book_viewing"):
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_023", "time_text": "Saturday 11am"}}
        return None

    if scenario_id == "task_visit_single_hidden_flex":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("contact_poster") and not has("book_viewing") and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "No Tuesday slot matches. I can offer Saturday 1pm or Sunday 5pm instead."}
        if not has("contact_poster"):
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_023", "time_text": "Sunday 5pm"}}
        if not has("book_viewing"):
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_023", "time_text": "Sunday 5pm"}}
        return None

    if scenario_id == "task_visit_multi":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not selected_posts and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "I shortlisted post_031 at tomorrow 7pm and post_052 at Sunday 4pm. Which listings do you want to pursue?"}
        if "post_031" not in booked and count("contact_poster") == 0 and "confirm tomorrow 7pm" not in last_user_message:
            return {"action_type": "assistant_message", "assistant_message": "Please confirm tomorrow 7pm for post_031."}
        if "post_031" not in booked and count("contact_poster") == 0:
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_031", "time_text": "tomorrow 7pm"}}
        if "post_031" not in booked:
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_031", "time_text": "tomorrow 7pm"}}
        if "post_052" not in booked and count("contact_poster") == 1 and "confirm sunday 4pm" not in last_user_message:
            return {"action_type": "assistant_message", "assistant_message": "Please confirm Sunday 4pm for post_052."}
        if "post_052" not in booked and count("contact_poster") == 1 and count("book_viewing") == 1:
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_052", "time_text": "Sunday 4pm"}}
        if "post_052" not in booked:
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_052", "time_text": "Sunday 4pm"}}
        return None

    return None

def compact_observation(obs: dict[str, Any]) -> dict[str, Any]:
    return {
        "scenario_id": obs.get("scenario_id"),
        "phase": obs.get("phase"),
        "status": obs.get("status"),
        "buyer_profile_stored": obs.get("buyer_profile_stored", False),
        "seller_profile_stored": obs.get("seller_profile_stored", False),
        "last_user_message": obs.get("last_user_message"),
        "current_user_request": obs.get("current_user_request"),
        "available_tools": obs.get("available_tools", []),
        "remaining_required_fields": obs.get("remaining_required_fields", []),
        "prerequisites_satisfied": obs.get("prerequisites_satisfied", {}),
        "recent_tool_calls": obs.get("recent_tool_calls", []),
        "last_tool_result": obs.get("last_tool_result", {}),
        "selected_posts": obs.get("selected_posts", []),
        "violations": obs.get("violations", []),
        "booked_visits": obs.get("booked_visits", []),
        "feedback_summary": obs.get("feedback_summary", ""),
    }

def prompt_from_observation(obs: dict[str, Any]) -> str:
    return (
        "You are a broker policy for the Flatmate RL environment. "
        "Return exactly one JSON action and no extra text.\n\n"
        "Valid action shapes:\n"
        "{\"action_type\":\"assistant_message\",\"assistant_message\":\"...\"}\n"
        "{\"action_type\":\"tool_call\",\"tool_name\":\"...\",\"tool_arguments\":{...}}\n\n"
        f"Observation:\n{json.dumps(compact_observation(obs), ensure_ascii=False, sort_keys=True)}\n\n"
        "Action:\n"
    )


## Collect Step-Labeled Web Rows

With `SEEDS_PER_SCENARIO = 5`, step 1 gets five examples per scenario from different seeds. Later curriculum stages use rows where `step_number <= stage_step`.


In [5]:
@dataclass
class CurriculumCollectionConfig:
    seeds_per_scenario: int = SEEDS_PER_SCENARIO
    start_seed: int = START_SEED
    max_steps: int = MAX_COLLECT_STEPS

async def collect_curriculum_rows(config: CurriculumCollectionConfig = CurriculumCollectionConfig()) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for scenario_id in SCENARIOS:
        for seed_offset in range(config.seeds_per_scenario):
            seed = config.start_seed + seed_offset
            prefix_actions: list[dict[str, Any]] = []
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                for step_idx in range(config.max_steps):
                    expected = heuristic_action(obs)
                    if expected is None or obs.get("done"):
                        break
                    rows.append({
                        "prompt": prompt_from_observation(obs),
                        "scenario_id": scenario_id,
                        "seed": seed,
                        "episode_key": f"{scenario_id}:{seed}",
                        "step_idx": step_idx,
                        "step_number": step_idx + 1,
                        "prefix_actions_json": json.dumps(prefix_actions, ensure_ascii=False),
                        "reference_action_json": json.dumps(expected, ensure_ascii=False, sort_keys=True),
                    })
                    obs = await env.step(expected)
                    prefix_actions.append(expected)
                    if obs.get("done"):
                        break
            print(f"scenario={scenario_id} seed={seed} total_rows={len(rows)}")
    return rows

rows = await collect_curriculum_rows()
rows_df = pd.DataFrame(rows)
rows_df.to_csv(RESULTS_DIR / "curriculum_rows.csv", index=False)
print(rows_df.groupby(["scenario_id", "step_number"]).size().unstack(fill_value=0))

step1_examples = rows_df[rows_df["step_number"] == 1].sort_values(["scenario_id", "seed"])
display(step1_examples[["scenario_id", "seed", "step_number", "reference_action_json"]].groupby("scenario_id").head(5))

steps_1_to_2_preview = rows_df[rows_df["step_number"] <= 2].sort_values(["scenario_id", "seed", "step_number"])
display(steps_1_to_2_preview[["scenario_id", "seed", "step_number", "reference_action_json"]].head(20))

train_dataset_all = Dataset.from_list([
    {**row, "prompt": [{"role": "user", "content": row["prompt"]}]}
    for row in rows
])
train_dataset_all


scenario=task_visit_single seed=101 total_rows=9
scenario=task_visit_single seed=102 total_rows=18
scenario=task_visit_single seed=103 total_rows=27
scenario=task_visit_single seed=104 total_rows=36
scenario=task_visit_single seed=105 total_rows=45
scenario=task_visit_single_hidden_flex seed=101 total_rows=54
scenario=task_visit_single_hidden_flex seed=102 total_rows=63
scenario=task_visit_single_hidden_flex seed=103 total_rows=72
scenario=task_visit_single_hidden_flex seed=104 total_rows=81
scenario=task_visit_single_hidden_flex seed=105 total_rows=90
scenario=task_visit_multi seed=101 total_rows=103
scenario=task_visit_multi seed=102 total_rows=116
scenario=task_visit_multi seed=103 total_rows=129
scenario=task_visit_multi seed=104 total_rows=142
scenario=task_visit_multi seed=105 total_rows=155
scenario=task_visit_single_seller_followup seed=101 total_rows=166
scenario=task_visit_single_seller_followup seed=102 total_rows=177
scenario=task_visit_single_seller_followup seed=103 total

,scenario_id,seed,step_number,reference_action_json
90,task_visit_multi,101,1,"{""action_type"": ""assistant_message"", ""assistan..."
103,task_visit_multi,102,1,"{""action_type"": ""assistant_message"", ""assistan..."
116,task_visit_multi,103,1,"{""action_type"": ""assistant_message"", ""assistan..."
129,task_visit_multi,104,1,"{""action_type"": ""assistant_message"", ""assistan..."
142,task_visit_multi,105,1,"{""action_type"": ""assistant_message"", ""assistan..."
0,task_visit_single,101,1,"{""action_type"": ""assistant_message"", ""assistan..."
9,task_visit_single,102,1,"{""action_type"": ""assistant_message"", ""assistan..."
18,task_visit_single,103,1,"{""action_type"": ""assistant_message"", ""assistan..."
27,task_visit_single,104,1,"{""action_type"": ""assistant_message"", ""assistan..."
36,task_visit_single,105,1,"{""action_type"": ""assistant_message"", ""assistan..."


,scenario_id,seed,step_number,reference_action_json
90,task_visit_multi,101,1,"{""action_type"": ""assistant_message"", ""assistan..."
91,task_visit_multi,101,2,"{""action_type"": ""tool_call"", ""tool_arguments"":..."
103,task_visit_multi,102,1,"{""action_type"": ""assistant_message"", ""assistan..."
104,task_visit_multi,102,2,"{""action_type"": ""tool_call"", ""tool_arguments"":..."
116,task_visit_multi,103,1,"{""action_type"": ""assistant_message"", ""assistan..."
117,task_visit_multi,103,2,"{""action_type"": ""tool_call"", ""tool_arguments"":..."
129,task_visit_multi,104,1,"{""action_type"": ""assistant_message"", ""assistan..."
130,task_visit_multi,104,2,"{""action_type"": ""tool_call"", ""tool_arguments"":..."
142,task_visit_multi,105,1,"{""action_type"": ""assistant_message"", ""assistan..."
143,task_visit_multi,105,2,"{""action_type"": ""tool_call"", ""tool_arguments"":..."


Dataset({
    features: ['prompt', 'scenario_id', 'seed', 'episode_key', 'step_idx', 'step_number', 'prefix_actions_json', 'reference_action_json'],
    num_rows: 210
})

## Parsing, Matching, and Metrics

`steps_to_correct_response` means the first generation attempt, from 1 to `attempts`, that exactly matches the expected action. If no attempt matches, it is empty.


In [6]:
def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return str(completion[0].get("content", ""))
    return str(completion)

def parse_json_action(text: Any) -> dict[str, Any]:
    text = completion_text(text).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("completion does not contain a JSON object")
    action = json.loads(text[start : end + 1])
    if action.get("action_type") == "assistant_message":
        msg = str(action.get("assistant_message", "")).strip()
        if not msg:
            raise ValueError("assistant_message action needs assistant_message")
        return {"action_type": "assistant_message", "assistant_message": msg}
    if action.get("action_type") == "tool_call":
        tool_name = str(action.get("tool_name", "")).strip()
        if not tool_name:
            raise ValueError("tool_call action needs tool_name")
        args = action.get("tool_arguments", {})
        if not isinstance(args, dict):
            raise ValueError("tool_arguments must be an object")
        return {"action_type": "tool_call", "tool_name": tool_name, "tool_arguments": args}
    raise ValueError("action_type must be assistant_message or tool_call")

def normalize_text(text: str) -> str:
    return " ".join(str(text).lower().replace("-", " ").split())

def normalize_action(action: dict[str, Any]) -> dict[str, Any]:
    if action.get("action_type") == "assistant_message":
        return {
            "action_type": "assistant_message",
            "assistant_message": normalize_text(str(action.get("assistant_message", ""))),
        }
    return {
        "action_type": "tool_call",
        "tool_name": str(action.get("tool_name", "")),
        "tool_arguments": action.get("tool_arguments", {}) if isinstance(action.get("tool_arguments", {}), dict) else {},
    }

def assistant_required_checks(expected_message: str) -> dict[str, list[str]]:
    """Build semantic checks from the expected assistant message.

    Exact string matching is too brittle for generated assistant text, but
    accepting any assistant_message is too weak. These checks make step-1
    correctness concrete: if the expected message asks for diet and visit
    availability, the generated message must ask for both concepts.
    """
    msg = normalize_text(expected_message)
    checks: dict[str, list[str]] = {}
    if "diet" in msg or "dietary" in msg:
        checks["diet"] = ["diet", "dietary", "food", "veg", "vegetarian", "non veg", "non vegetarian"]
    if "availability" in msg or "available" in msg or "slot" in msg or "time" in msg:
        checks["visit_availability"] = ["availability", "available", "time", "times", "slot", "slots", "visit", "when"]
    if "household" in msg:
        checks["household"] = ["household", "flat", "home"]
    if "who the flat is for" in msg or "occupation" in msg:
        checks["occupation_requirement"] = ["who the flat is for", "occupation", "professional", "working", "student"]
    if "confirm" in msg:
        checks["confirmation"] = ["confirm", "confirmation", "works", "okay", "ok"]

    for token in ["post_023", "post_031", "post_052", "post_dynamic_followup_1"]:
        if token in msg:
            checks[f"mentions_{token}"] = [token]
    for token in ["saturday", "sunday", "tomorrow", "today", "tuesday", "11am", "1pm", "4pm", "5pm", "7pm"]:
        if token in msg:
            checks[f"mentions_{token}"] = [token]
    return checks

def assistant_intent_match(got_message: str, expected_message: str) -> bool:
    checks = assistant_required_checks(expected_message)
    got = normalize_text(got_message)
    if not checks:
        return bool(got.strip())
    return all(any(keyword in got for keyword in keywords) for keywords in checks.values())

def assistant_intent_score(got_message: str, expected_message: str) -> float:
    checks = assistant_required_checks(expected_message)
    got = normalize_text(got_message)
    if not checks:
        return 1.0 if got.strip() else 0.0
    passed = sum(1 for keywords in checks.values() if any(keyword in got for keyword in keywords))
    return passed / max(1, len(checks))

def action_match_parts(got: dict[str, Any], expected: dict[str, Any]) -> dict[str, bool]:
    got_n = normalize_action(got)
    exp_n = normalize_action(expected)
    action_type_match = got_n.get("action_type") == exp_n.get("action_type")
    tool_name_match = action_type_match and got_n.get("action_type") == "tool_call" and got_n.get("tool_name") == exp_n.get("tool_name")
    args_match = tool_name_match and got_n.get("tool_arguments") == exp_n.get("tool_arguments")
    assistant_match = action_type_match and got_n.get("action_type") == "assistant_message" and got_n.get("assistant_message") == exp_n.get("assistant_message")
    assistant_intent = action_type_match and got_n.get("action_type") == "assistant_message" and assistant_intent_match(got_n.get("assistant_message", ""), exp_n.get("assistant_message", ""))
    valid_response_match = args_match or assistant_intent
    exact_match = args_match or assistant_match
    return {
        "exact_match": exact_match,
        "valid_response_match": valid_response_match,
        "action_type_match": action_type_match,
        "tool_name_match": tool_name_match,
        "args_match": args_match,
        "assistant_match": assistant_match,
        "assistant_intent_match": assistant_intent,
    }

def expected_action_score(got: dict[str, Any], expected: dict[str, Any]) -> float:
    parts = action_match_parts(got, expected)
    if parts["valid_response_match"]:
        return 1.0
    if expected.get("action_type") == "assistant_message" and got.get("action_type") == "assistant_message":
        return 0.1 + 0.8 * assistant_intent_score(str(got.get("assistant_message", "")), str(expected.get("assistant_message", "")))
    score = 0.0
    if parts["action_type_match"]:
        score += 0.1
    if parts["tool_name_match"]:
        score += 0.4
    return score

def summarize_step_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    if eval_df.empty:
        return eval_df
    return (
        eval_df.groupby(["stage", "scenario_id", "step_number"], as_index=False)
        .agg(
            rows=("exact_match", "size"),
            json_ok_rate=("json_ok", "mean"),
            exact_match_rate=("exact_match", "mean"),
            valid_response_match_rate=("valid_response_match", "mean"),
            action_type_match_rate=("action_type_match", "mean"),
            tool_name_match_rate=("tool_name_match", "mean"),
            assistant_intent_match_rate=("assistant_intent_match", "mean"),
            avg_steps_to_correct=("steps_to_correct_response", "mean"),
            missing_correct_rate=("steps_to_correct_response", lambda s: s.isna().mean()),
        )
        .sort_values(["stage", "scenario_id", "step_number"])
    )


## First-Step Validity Sanity Check

Run this before GRPO if step 1 is failing. It verifies that the expected-response validator gives high score only to responses that ask for the required first-step information.

In [7]:

def first_step_validity_sanity_check() -> pd.DataFrame:
    step1 = rows_df[rows_df["step_number"] == 1].sort_values(["scenario_id", "seed"]).iloc[0]
    expected = json.loads(step1["reference_action_json"])
    candidates = [
        expected,
        {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference and visit availability."},
        {"action_type": "assistant_message", "assistant_message": "Please share your budget."},
        {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference."},
        {"action_type": "assistant_message", "assistant_message": "When can you visit the flat?"},
        {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}},
    ]
    records = []
    for candidate in candidates:
        parts = action_match_parts(candidate, expected)
        records.append({
            "scenario_id": step1["scenario_id"],
            "seed": step1["seed"],
            "expected": json.dumps(expected, ensure_ascii=False, sort_keys=True),
            "candidate": json.dumps(candidate, ensure_ascii=False, sort_keys=True),
            "expected_action_score_0_to_1": expected_action_score(candidate, expected),
            "grpo_expected_reward": 2.0 * expected_action_score(candidate, expected) - 0.5,
            **parts,
        })
    return pd.DataFrame(records)

first_step_validity_sanity_check()


,scenario_id,seed,expected,candidate,expected_action_score_0_to_1,grpo_expected_reward,exact_match,valid_response_match,action_type_match,tool_name_match,args_match,assistant_match,assistant_intent_match
0,task_visit_multi,101,"{""action_type"": ""assistant_message"", ""assistan...","{""action_type"": ""assistant_message"", ""assistan...",1.0,1.5,True,True,True,False,False,True,True
1,task_visit_multi,101,"{""action_type"": ""assistant_message"", ""assistan...","{""action_type"": ""assistant_message"", ""assistan...",1.0,1.5,True,True,True,False,False,True,True
2,task_visit_multi,101,"{""action_type"": ""assistant_message"", ""assistan...","{""action_type"": ""assistant_message"", ""assistan...",0.1,-0.3,False,False,True,False,False,False,False
3,task_visit_multi,101,"{""action_type"": ""assistant_message"", ""assistan...","{""action_type"": ""assistant_message"", ""assistan...",0.5,0.5,False,False,True,False,False,False,False
4,task_visit_multi,101,"{""action_type"": ""assistant_message"", ""assistan...","{""action_type"": ""assistant_message"", ""assistan...",0.5,0.5,False,False,True,False,False,False,False
5,task_visit_multi,101,"{""action_type"": ""assistant_message"", ""assistan...","{""action_type"": ""tool_call"", ""tool_arguments"":...",0.0,-0.5,False,False,False,False,False,False,False


## Reward Functions

The expected-action reward directly optimizes the metrics you asked to track. The endpoint reward keeps the policy grounded in real web-environment consequences.


In [8]:
REWARD_DEBUG = {"parse_errors": [], "endpoint_errors": [], "prefix_done": 0, "valid": 0, "invalid": 0}

JSON_VALID_REWARD = 0.2
JSON_INVALID_REWARD = -1.2
EXPECTED_REWARD_SCALE = 3.0
EXPECTED_REWARD_BIAS = -1.0
ENDPOINT_REWARD_WEIGHT = 0.35


def _validate_reward_length(name: str, rewards: list[float], completions: list[Any]) -> list[float]:
    if len(rewards) != len(completions):
        raise ValueError(f"{name} length mismatch: {len(rewards)} != {len(completions)}")
    return rewards


def json_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        try:
            parse_json_action(completion_text(completion))
            rewards.append(JSON_VALID_REWARD)
            REWARD_DEBUG["valid"] += 1
        except Exception as exc:
            rewards.append(JSON_INVALID_REWARD)
            REWARD_DEBUG["invalid"] += 1
            if len(REWARD_DEBUG["parse_errors"]) < 20:
                REWARD_DEBUG["parse_errors"].append({"error": str(exc), "completion": completion_text(completion)[:240]})
    return _validate_reward_length("json_format_reward", rewards, completions)


def expected_action_reward(completions, reference_action_json, **kwargs) -> list[float]:
    rewards = []
    for completion, reference_json in zip(completions, reference_action_json):
        try:
            got = parse_json_action(completion_text(completion))
            expected = json.loads(reference_json)
            score = expected_action_score(got, expected)
            rewards.append(EXPECTED_REWARD_SCALE * score + EXPECTED_REWARD_BIAS)
        except Exception:
            rewards.append(-1.0)
    return _validate_reward_length("expected_action_reward", rewards, completions)


async def score_one_completion(completion: Any, scenario_id: str, seed: int, prefix_actions_json: str) -> float:
    try:
        action = parse_json_action(completion_text(completion))
        prefix_actions = json.loads(prefix_actions_json)
    except Exception:
        return -1.0

    try:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=int(seed))
            for prefix_action in prefix_actions:
                obs = await env.step(prefix_action)
                if obs.get("done"):
                    REWARD_DEBUG["prefix_done"] += 1
                    return -0.1

            before_violations = len(obs.get("violations", []))
            before_bookings = len(obs.get("booked_visits", []))
            obs = await env.step(action)

        reward = float(obs.get("reward") or obs.get("step_reward") or 0.0)
        reward += 0.15
        if len(obs.get("violations", [])) > before_violations:
            reward -= 0.75
        if len(obs.get("booked_visits", [])) > before_bookings:
            reward += 1.0
        if obs.get("done") and obs.get("status") != "failed":
            reward += 0.5
        reward = float(max(-2.0, min(2.0, reward)))
        return ENDPOINT_REWARD_WEIGHT * reward
    except Exception as exc:
        if len(REWARD_DEBUG["endpoint_errors"]) < 10:
            REWARD_DEBUG["endpoint_errors"].append(repr(exc))
        return -1.0


async def _score_with_semaphore(semaphore, completion, scenario_id, seed, prefix_actions_json) -> float:
    async with semaphore:
        return await score_one_completion(completion, scenario_id, seed, prefix_actions_json)


async def score_completion_batch(completions, scenario_id, seed, prefix_actions_json) -> list[float]:
    # The Space is configured for 4 concurrent envs. Use 3 reward sessions to leave one spare.
    semaphore = asyncio.Semaphore(3)
    tasks = [
        _score_with_semaphore(semaphore, c, s, int(sd), p)
        for c, s, sd, p in zip(completions, scenario_id, seed, prefix_actions_json)
    ]
    rewards = list(await asyncio.gather(*tasks))
    return _validate_reward_length("score_completion_batch", rewards, completions)


def run_async_blocking(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, Any] = {}

    def runner():
        try:
            result["value"] = asyncio.run(coro)
        except Exception as exc:
            result["error"] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if "error" in result:
        raise result["error"]
    return result["value"]


def endpoint_transition_reward(completions, scenario_id, seed, prefix_actions_json, **kwargs) -> list[float]:
    rewards = run_async_blocking(score_completion_batch(completions, scenario_id, seed, prefix_actions_json))
    return _validate_reward_length("endpoint_transition_reward", rewards, completions)


## Model Loading

This keeps learning from the prior notebook when a checkpoint exists. Preference order:

1. previous GRPO adapter: `flatmate-rl-grpo-policy`
2. previous SFT adapter: `flatmate-rl-grpo-policy/sft_warmup`
3. base Qwen model


In [9]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_merged_start_model():
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)
    for adapter_dir in [PREVIOUS_GRPO_DIR, PREVIOUS_SFT_DIR]:
        if not adapter_dir:
            continue
        adapter_path = Path(adapter_dir)
        if not (adapter_path / "adapter_config.json").exists():
            continue
        try:
            print("Starting from adapter:", adapter_dir)
            model = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
            if hasattr(model, "peft_config"):
                delattr(model, "peft_config")
            return model
        except Exception as exc:
            print(f"Skipping adapter {adapter_dir!r} due to load error: {exc}")
    print("Starting from base model:", MODEL_NAME)
    return base


## SFT Step Warmup

This optional warmup teaches the model to emit the expected action format for the collected step examples before GRPO starts optimizing rewards.


In [10]:
def make_sft_config(**kwargs):
    valid = set(inspect.signature(SFTConfig.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported SFTConfig fields:", skipped)
    return SFTConfig(**filtered)

def make_sft_trainer(**kwargs):
    valid = set(inspect.signature(SFTTrainer.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported SFTTrainer fields:", skipped)
    return SFTTrainer(**filtered)

sft_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "completion": [{"role": "assistant", "content": row["reference_action_json"]}],
    }
    for row in rows
])

sft_args = make_sft_config(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=2,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    max_length=SFT_MAX_LENGTH,
    completion_only_loss=True,
    report_to="none",
)

sft_start_model = load_merged_start_model()
sft_trainer = make_sft_trainer(
    model=sft_start_model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    tokenizer=tokenizer,
    peft_config=peft_config,
)
sft_trainer.train()
sft_trainer.save_model(SFT_OUTPUT_DIR)
print("Saved step SFT warmup to", SFT_OUTPUT_DIR)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Starting from adapter: flatmate-rl-grpo-policy
Skipping unsupported SFTTrainer fields: ['tokenizer']


Tokenizing train dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
2,0.990262
4,0.860283
6,0.457809
8,0.364143
10,0.217542
12,0.300292
14,0.398058
16,0.339035
18,0.202766
20,0.192734


Saved step SFT warmup to flatmate-rl-grpo-2-step-curriculum/sft_step_warmup


## Evaluation: Expected vs Got

Run this before and after each curriculum stage. It records the expected action, generated action, parse status, match fields, and `steps_to_correct_response`.


In [11]:
def load_adapter_for_eval(adapter_dir: str):
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    model = AutoPeftModelForCausalLM.from_pretrained(
        adapter_dir,
        device_map="auto",
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )
    model.eval()
    model.config.use_cache = False
    return model

def build_inputs(prompt: str, model):
    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return tokenizer(chat_text, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to(model.device)

def generate_once(model, prompt: str, sample: bool, temperature: float = 0.7) -> str:
    inputs = build_inputs(prompt, model)
    generation_kwargs = {
        "max_new_tokens": MAX_COMPLETION_LENGTH,
        "do_sample": sample,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if sample:
        generation_kwargs.update({"temperature": temperature, "top_p": 0.95})
    with torch.no_grad():
        output = model.generate(**inputs, **generation_kwargs)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def evaluate_expected_vs_got(model, eval_rows: list[dict[str, Any]], stage: str, attempts: int = 5) -> pd.DataFrame:
    records = []
    progress = tqdm(eval_rows, desc=f"eval {stage}", total=len(eval_rows), dynamic_ncols=True)
    for row in progress:
        progress.set_postfix({"scenario": row["scenario_id"], "seed": row["seed"], "step": row["step_number"]})
        expected = json.loads(row["reference_action_json"])
        first_raw = ""
        first_error = ""
        first_got: dict[str, Any] | None = None
        first_parts = {"exact_match": False, "valid_response_match": False, "action_type_match": False, "tool_name_match": False, "args_match": False, "assistant_match": False, "assistant_intent_match": False}
        steps_to_correct = None

        for attempt_idx in range(1, attempts + 1):
            raw = generate_once(model, row["prompt"], sample=(attempt_idx > 1))
            if attempt_idx == 1:
                first_raw = raw
            try:
                got = parse_json_action(raw)
                parts = action_match_parts(got, expected)
                if attempt_idx == 1:
                    first_got = got
                    first_parts = parts
                if parts["valid_response_match"] and steps_to_correct is None:
                    steps_to_correct = attempt_idx
                    break
            except Exception as exc:
                if attempt_idx == 1:
                    first_error = str(exc)

        records.append({
            "stage": stage,
            "scenario_id": row["scenario_id"],
            "seed": row["seed"],
            "episode_key": row["episode_key"],
            "step_number": row["step_number"],
            "expected_action_json": row["reference_action_json"],
            "got_action_json": json.dumps(first_got, ensure_ascii=False, sort_keys=True) if first_got else "",
            "raw_completion": first_raw,
            "json_ok": first_got is not None,
            "parse_error": first_error,
            "steps_to_correct_response": steps_to_correct,
            **first_parts,
        })
    return pd.DataFrame(records)

# Evaluate the SFT warmup before GRPO curriculum.
eval_model = load_adapter_for_eval(SFT_OUTPUT_DIR)
baseline_eval_df = evaluate_expected_vs_got(eval_model, rows, stage="sft_warmup", attempts=EVAL_ATTEMPTS)
baseline_eval_df.to_csv(RESULTS_DIR / "eval_sft_warmup.csv", index=False)
baseline_summary = summarize_step_metrics(baseline_eval_df)
baseline_summary.to_csv(RESULTS_DIR / "summary_sft_warmup.csv", index=False)

# Release eval model memory before GRPO stage loading.
del eval_model
if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()

baseline_summary.head(20)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

eval sft_warmup:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,stage,scenario_id,step_number,rows,json_ok_rate,exact_match_rate,valid_response_match_rate,action_type_match_rate,tool_name_match_rate,assistant_intent_match_rate,avg_steps_to_correct,missing_correct_rate
0,sft_warmup,task_visit_multi,1,5,1.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0
1,sft_warmup,task_visit_multi,2,5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
2,sft_warmup,task_visit_multi,3,5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
3,sft_warmup,task_visit_multi,4,5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
4,sft_warmup,task_visit_multi,5,5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
5,sft_warmup,task_visit_multi,6,5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
6,sft_warmup,task_visit_multi,7,5,1.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0
7,sft_warmup,task_visit_multi,8,5,1.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0
8,sft_warmup,task_visit_multi,9,5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0
9,sft_warmup,task_visit_multi,10,5,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0


## Curriculum GRPO

Stage 1 trains on only `step_number <= 1`. Stage 2 trains on `step_number <= 2`, and so on. The adapter from the previous stage becomes the starting point for the next stage.


In [ ]:
def make_grpo_config(**kwargs):
    valid = set(inspect.signature(GRPOConfig.__init__).parameters)
    normalized = dict(kwargs)

    if "max_prompt_length" in normalized and "max_prompt_length" not in valid and "max_length" in valid:
        normalized["max_length"] = normalized.pop("max_prompt_length")
        print("GRPOConfig compatibility: mapped max_prompt_length -> max_length")

    filtered = {k: v for k, v in normalized.items() if k in valid}
    skipped = sorted(set(normalized) - set(filtered))

    critical_keys = {"learning_rate", "num_generations", "max_completion_length", "max_steps"}
    if "max_prompt_length" in kwargs:
        if "max_prompt_length" in valid:
            critical_keys.add("max_prompt_length")
        elif "max_length" in valid:
            critical_keys.add("max_length")

    missing_critical = sorted(key for key in critical_keys if key in normalized and key not in filtered)
    if missing_critical:
        raise ValueError(f"Critical GRPOConfig fields unsupported in installed TRL: {missing_critical}")

    if skipped:
        print("Skipping non-critical GRPOConfig fields:", skipped)
    return GRPOConfig(**filtered)


def dataset_for_stage(max_step_number: int) -> Dataset:
    stage_rows = [row for row in rows if int(row["step_number"]) <= int(max_step_number)]
    return Dataset.from_list([
        {**row, "prompt": [{"role": "user", "content": row["prompt"]}]}
        for row in stage_rows
    ])


def load_stage_start_model(adapter_dir: str):
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
    if hasattr(model, "peft_config"):
        delattr(model, "peft_config")
    return model


all_eval_dfs = [baseline_eval_df]
all_summaries = [baseline_summary]
current_adapter_dir = SFT_OUTPUT_DIR

for stage_step in range(1, MAX_CURRICULUM_STEP + 1):
    stage_name = f"steps_1_to_{stage_step}"
    stage_output_dir = f"{OUTPUT_DIR}/{stage_name}"
    print("\n=== GRPO curriculum stage:", stage_name, "===")

    stage_dataset = dataset_for_stage(stage_step)
    print("stage rows:", len(stage_dataset))

    stage_model = load_stage_start_model(current_adapter_dir)
    grpo_args = make_grpo_config(
        output_dir=stage_output_dir,
        learning_rate=1e-5,
        per_device_train_batch_size=GRPO_BATCH_SIZE,
        gradient_accumulation_steps=GRPO_GRAD_ACCUM,
        num_generations=NUM_GENERATIONS,
        max_prompt_length=MAX_PROMPT_LENGTH,
        max_completion_length=MAX_COMPLETION_LENGTH,
        max_steps=GRPO_STEPS_PER_STAGE,
        logging_steps=1,
        save_steps=max(10, GRPO_STEPS_PER_STAGE),
        save_total_limit=1,
        beta=GRPO_BETA,
        gradient_checkpointing=True,
        report_to="none",
        bf16=use_bf16,
        fp16=torch.cuda.is_available() and not use_bf16,
    )

    REWARD_DEBUG.update({"parse_errors": [], "endpoint_errors": [], "prefix_done": 0, "valid": 0, "invalid": 0})
    trainer = GRPOTrainer(
        model=stage_model,
        processing_class=tokenizer,
        reward_funcs=[expected_action_reward, endpoint_transition_reward, json_format_reward],
        args=grpo_args,
        train_dataset=stage_dataset,
        peft_config=peft_config,
    )
    train_result = trainer.train()
    trainer.save_model(stage_output_dir)
    current_adapter_dir = stage_output_dir

    print("Reward debug:", REWARD_DEBUG)
    (Path(stage_output_dir) / "train_log_history.json").write_text(json.dumps(trainer.state.log_history, indent=2))

    eval_model = load_adapter_for_eval(stage_output_dir)
    eval_df = evaluate_expected_vs_got(eval_model, rows, stage=stage_name, attempts=EVAL_ATTEMPTS)
    summary_df = summarize_step_metrics(eval_df)
    eval_df.to_csv(RESULTS_DIR / f"eval_{stage_name}.csv", index=False)
    summary_df.to_csv(RESULTS_DIR / f"summary_{stage_name}.csv", index=False)
    all_eval_dfs.append(eval_df)
    all_summaries.append(summary_df)

    display(summary_df[summary_df["step_number"] <= stage_step].head(40))

    # Keep only one large model in GPU memory at a time.
    del trainer
    del eval_model
    del stage_model
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()

combined_eval_df = pd.concat(all_eval_dfs, ignore_index=True)
combined_summary_df = pd.concat(all_summaries, ignore_index=True)
combined_eval_df.to_csv(RESULTS_DIR / "eval_all_stages.csv", index=False)
combined_summary_df.to_csv(RESULTS_DIR / "summary_all_stages.csv", index=False)
combined_summary_df.tail(30)



=== GRPO curriculum stage: steps_1_to_1 ===
stage rows: 20


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Skipping unsupported GRPOConfig fields: ['max_prompt_length']


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,-0.019164
2,0.011030
3,0.036010
4,0.040724
5,-0.019118
6,-0.019079
7,-0.018994
8,-0.019637
9,0.020600
10,0.032920


...


=== GRPO curriculum stage: steps_1_to_4 ===
stage rows: 80


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Skipping unsupported GRPOConfig fields: ['max_prompt_length']


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,-0.226530
2,-0.272049
3,-0.434403
4,0.370551
5,-0.007279
6,-0.100892
7,0.075259
8,0.014917
9,0.195243
10,-0.249758


## Accuracy Plots

These plots show whether each stage improves exact-match accuracy and reduces attempts needed to produce the correct response.


In [ ]:
import matplotlib.pyplot as plt

plot_df = combined_summary_df.copy()
plot_df["stage_order"] = plot_df["stage"].map({stage: idx for idx, stage in enumerate(plot_df["stage"].drop_duplicates())})

for scenario_id in SCENARIOS:
    sub = plot_df[plot_df["scenario_id"] == scenario_id]
    if sub.empty:
        continue
    pivot = sub.pivot_table(index="stage", columns="step_number", values="valid_response_match_rate", aggfunc="mean")
    ax = pivot.plot(marker="o", figsize=(10, 4), title=f"Valid response match by step: {scenario_id}")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("valid response match rate")
    plt.show()

attempts_pivot = combined_summary_df.pivot_table(index="stage", columns="step_number", values="avg_steps_to_correct", aggfunc="mean")
ax = attempts_pivot.plot(marker="o", figsize=(10, 4), title="Average attempts to correct response")
ax.grid(True, alpha=0.3)
ax.set_ylabel("attempts; lower is better")
plt.show()


## Inspect Mistakes

Use this to see expected response vs got response for any stage and step.


In [ ]:
def show_mistakes(stage: str, step_number: int, limit: int = 20):
    df = combined_eval_df[
        (combined_eval_df["stage"] == stage)
        & (combined_eval_df["step_number"] == step_number)
        & (~combined_eval_df["valid_response_match"])
    ].copy()
    cols = [
        "scenario_id",
        "seed",
        "step_number",
        "steps_to_correct_response",
        "valid_response_match",
        "expected_action_json",
        "got_action_json",
        "parse_error",
        "raw_completion",
    ]
    return df[cols].head(limit)

show_mistakes(stage=f"steps_1_to_{MAX_CURRICULUM_STEP}", step_number=1, limit=10)
